# 02 - Persian Text Preprocessing
Turn raw Persian reviews into clean text a model can learn from:
normalization, noise removal, a **sentiment-aware** stopword list, and a fixed train / validation / test split.

## 0. Setup

In [1]:
!pip install -q hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
BASE_DIR = "/content/drive/MyDrive/snappfood-sentiment"
DATA_PATH = f"{BASE_DIR}/Snappfood - Sentiment Analysis.csv"
OUT_DIR = f"{BASE_DIR}/processed"
os.makedirs(OUT_DIR, exist_ok=True)

print("Data file found:", os.path.exists(DATA_PATH))

Mounted at /content/drive
Data file found: True


In [3]:
import re
import warnings
from collections import Counter

import pandas as pd
from hazm import Normalizer, stopwords_list
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 150)
warnings.filterwarnings("ignore", message="This pattern")

## 1. Load data
Same as notebook 01. We already know all 70,000 rows are valid.

In [4]:
raw = pd.read_csv(DATA_PATH, sep="\t", on_bad_lines="skip", engine="python")

df = raw[["comment", "label"]].copy()
df["label"] = df["label"].str.strip().str.upper()
df["sentiment"] = df["label"].map({"HAPPY": 1, "SAD": 0})
print(df.shape)
df.head()

(70000, 3)


,comment,label,sentiment
0,واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح,SAD,0
1,قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از موقع رسید، شما ببین چقدرررررررررررر پلاک خفنههههه، من سالهاست مشتریشونم و سالهاست مزه بهشت میده غذاشون,HAPPY,1
2,قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط ظاهر فریبنده داره، پرش میکنن کالباس و قارچ,SAD,0
3,عالللی بود همه چه درست و به اندازه و کیفیت خوب، امیداورم همیشه کیفیتتون خوب باشه ما مشتری همیشگی بشیم,HAPPY,1
4,شیرینی وانیلی فقط یک مدل بود.,HAPPY,1


## 2. What kind of noise do we have?
Before cleaning, let's count what we are actually dealing with.

In [5]:
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF]")

checks = {
    "URL":                    r"https?://|www\.",
    "English letters":        r"[a-zA-Z]",
    "Digits (any script)":    r"[0-9\u06F0-\u06F9\u0660-\u0669]",
    "Arabic ي or ك":          r"[\u064A\u0643]",
    "Repeated letters (3+)":  r"(.)\1{2,}",
    "Emoji":                  EMOJI_RE.pattern,
}

noise = pd.DataFrame({
    name: [df["comment"].str.contains(p, regex=True).sum()] for name, p in checks.items()
}, index=["comments"]).T
noise["percent"] = (noise["comments"] / len(df) * 100).round(2)
noise

,comments,percent
URL,0,0.00
English letters,365,0.52
Digits (any script),6630,9.47
Arabic ي or ك,0,0.00
Repeated letters (3+),4793,6.85
Emoji,0,0.00


In [6]:
# a few examples with repeated letters and emojis
print(df.loc[df["comment"].str.contains(r"(.)\1{2,}", regex=True), "comment"].head(5).to_string())
print()
print(df.loc[df["comment"].str.contains(EMOJI_RE.pattern, regex=True), "comment"].head(5).to_string())

1     قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از موقع رسید، شما ببین چقدرررررررررررر پلاک خفنههههه، من سالهاست مشتریشونم و سالهاست مزه بهشت میده غذاشون
3                                                  عالللی بود همه چه درست و به اندازه و کیفیت خوب، امیداورم همیشه کیفیتتون خوب باشه ما مشتری همیشگی بشیم
8                                                                                        همه اقلام تازه و به روز وخیلیییییی سریع بدستم رسید واقعا متشکرم
9                                                                                          همه چی خوب ولی هات داگ دورش کلا سوخته بود و داخلش خام بود!!!!
12                                                                                                    مثل همیشه عااااالی هستی پرپروک جاااااان مچکریم ازت

Series([], )


## 3. The stopword trap
Hazm ships a ready-made Persian stopword list, but it was built for **news text**, not reviews.
Let's check whether it contains words that clearly carry sentiment.

In [7]:
hazm_stopwords = set(stopwords_list())
print("hazm stopwords:", len(hazm_stopwords))

test_words = ["عالی", "خوب", "بهترین", "متاسفانه", "نبود", "نیست", "نه", "هیچ", "ولی", "خیلی", "کم"]
for w in test_words:
    print(f"{w:10} in hazm stopwords? {w in hazm_stopwords}")

hazm stopwords: 389
عالی       in hazm stopwords? True
خوب        in hazm stopwords? True
بهترین     in hazm stopwords? True
متاسفانه   in hazm stopwords? True
نبود       in hazm stopwords? True
نیست       in hazm stopwords? True
نه         in hazm stopwords? True
هیچ        in hazm stopwords? True
ولی        in hazm stopwords? True
خیلی       in hazm stopwords? True
کم         in hazm stopwords? True


In [8]:
# how many reviews would lose an important word if we used the list as-is?
for w in ["عالی", "خوب", "نبود", "متاسفانه"]:
    n = df["comment"].str.contains(rf"\b{w}\b", regex=True).sum()
    print(f"{w:10} appears in {n:,} reviews")

عالی       appears in 10,990 reviews
خوب        appears in 12,457 reviews
نبود       appears in 7,985 reviews
متاسفانه   appears in 3,139 reviews


Removing «عالی» (excellent) or «نبود» (wasn't) from a sentiment dataset would throw away the most useful signal.
So we **keep** negations, contrast words, intensifiers and evaluative words, and only remove the truly neutral ones.

In [9]:
KEEP = {
    # negation
    "نه", "نبود", "نیست", "نیستند", "ندارد", "ندارند", "نباید", "نمی‌شود", "هیچ", "بدون", "عدم", "غیر", "بی",
    # contrast
    "اما", "ولی", "بلکه", "حتی", "فقط", "تنها",
    # intensity / amount
    "خیلی", "بسیار", "بسیاری", "کاملا", "کامل", "زیاد", "زیادی", "کم", "کمی", "بیش", "بیشتر", "پر", "کافی",
    # evaluation
    "عالی", "خوب", "خوبی", "بهتر", "بهترین", "مناسب", "متاسفانه", "جدی",
    # delivery / time (very relevant for food delivery)
    "رسید", "رسیدن", "دیر", "هنوز", "همیشه", "همواره", "دوباره",
}

STOPWORDS = hazm_stopwords - KEEP
print("final stopwords:", len(STOPWORDS))
print("rescued words:", sorted(hazm_stopwords & KEEP))

final stopwords: 343
rescued words: ['اما', 'بدون', 'بسیار', 'بسیاری', 'بلکه', 'بهتر', 'بهترین', 'بی', 'بیش', 'بیشتر', 'تنها', 'جدی', 'حتی', 'خوب', 'خوبی', 'خیلی', 'دوباره', 'رسید', 'رسیدن', 'زیاد', 'زیادی', 'عالی', 'عدم', 'غیر', 'فقط', 'متاسفانه', 'مناسب', 'نباید', 'نبود', 'ندارد', 'ندارند', 'نمی\u200cشود', 'نه', 'نیست', 'نیستند', 'همواره', 'همیشه', 'هنوز', 'هیچ', 'ولی', 'پر', 'کافی', 'کامل', 'کاملا', 'کم', 'کمی']


## 4. The cleaning function
Step by step:
1. lowercase English letters
2. remove URLs
3. hazm `Normalizer`: fixes Arabic ي/ك, spacing and half-spaces (نیم‌فاصله)
4. shorten repeated letters: «عالیییی» → «عالی»
5. keep emojis as separate tokens (😍 and 😡 are strong signals)
6. remove punctuation, digits and other symbols
7. drop stopwords and one-letter tokens

In [10]:
normalizer = Normalizer()

URL_RE = re.compile(r"https?://\S+|www\.\S+")
REPEAT_RE = re.compile(r"(.)\1{2,}")
PERSIAN = "\u0621-\u063A\u0641-\u064A\u067E\u0686\u0698\u06A9\u06AF\u06CC\u200c"
EMOJI_CHARS = "\U0001F300-\U0001FAFF\U00002600-\U000027BF"
NOT_ALLOWED_RE = re.compile(f"[^{PERSIAN}{EMOJI_CHARS}a-z\\s]")

def clean_text(text, stopwords=STOPWORDS):
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = normalizer.normalize(text)
    text = REPEAT_RE.sub(r"\1", text)
    text = EMOJI_RE.sub(lambda m: f" {m.group(0)} ", text)
    text = NOT_ALLOWED_RE.sub(" ", text)

    tokens = []
    for tok in text.split():
        tok = tok.strip("\u200c")
        if not tok or tok in stopwords:
            continue
        if len(tok) == 1 and not EMOJI_RE.match(tok):
            continue
        tokens.append(tok)
    return " ".join(tokens)

In [11]:
# test it on hand-written examples before running on all data
samples = [
    "غذا خيلي سرد بود و اصلا خوب نبود!!! 😡😡😡",
    "عالیییی بود ممنون از رستوران 😍",
    "پیتزا ۲ ساعت دیر رسید... www.snappfood.ir",
    "کیفیت غذا متوسط بود ولی پیک خیلی مودب بود",
]
for s in samples:
    print("before:", s)
    print("after: ", clean_text(s))
    print("naive: ", clean_text(s, stopwords=hazm_stopwords), "  <- with the original hazm list")
    print()

before: غذا خيلي سرد بود و اصلا خوب نبود!!! 😡😡😡
after:  غذا خیلی سرد اصلا خوب نبود 😡
naive:  غذا سرد اصلا 😡   <- with the original hazm list

before: عالیییی بود ممنون از رستوران 😍
after:  عالی ممنون رستوران 😍
naive:  ممنون رستوران 😍   <- with the original hazm list

before: پیتزا ۲ ساعت دیر رسید... www.snappfood.ir
after:  پیتزا ساعت دیر رسید
naive:  پیتزا ساعت دیر   <- with the original hazm list

before: کیفیت غذا متوسط بود ولی پیک خیلی مودب بود
after:  کیفیت غذا متوسط ولی پیک خیلی مودب
naive:  کیفیت غذا متوسط پیک مودب   <- with the original hazm list



## 5. Clean all reviews

In [12]:
%%time
df["clean_text"] = df["comment"].apply(clean_text)
df[["comment", "clean_text", "label"]].sample(10, random_state=1)

CPU times: user 23.8 s, sys: 0 ns, total: 23.8 s
Wall time: 24 s


,comment,clean_text,label
6670,به نسبت قیمت، حجم غذا و کیفیتش خیلی خوب بود. اگر جوجه کمی طعم دارتر میشد عالی بود. به موقع هم رسید.,قیمت حجم غذا کیفیتش خیلی خوب جوجه کمی طعم دارتر عالی موقع رسید,HAPPY
49567,افتضاح، اصلا قابل خوردن نبود، ریختیم دور، متاسفم,افتضاح اصلا خوردن نبود ریختیم متاسفم,SAD
50796,فوق العاده چرب بود و به شدت اذیت شدم و نیم خورده رها کردم,فوق‌العاده چرب شدت اذیت شدم نیم خورده رها,SAD
22310,به جای پنیر چرب روزانه پنیر کم چرب رژیمی فرستادن، اگر قراره هربار یک مغایرت باشه دیگه نمیشه اعتماد کرد که به پیک هاتون تاکید کنید که سفارش رو با ف...,پنیر چرب روزانه پنیر کم‌چرب رژیمی فرستادن قراره هربار مغایرت باشه دیگه نمیشه اعتماد پیک هاتون تاکید سفارش فاکتور چک لطفا,SAD
54037,چون با تخفیف گرفتم ایراد نداره ولی کلا روی پیتزا ۵تا ورق سوسیس بود.,تخفیف گرفتم ایراد نداره ولی کلا پیتزا ورق سوسیس,HAPPY
6869,فقط غذا رو‌ تو ظرف ریخته بودن فرستاده بودن نه قاشق چنگالی نه نونی نه یه تیکه پیازی کیفیت هم میتونست خیلی بیشتر از این باشه,فقط غذا ظرف ریخته فرستاده نه قاشق چنگالی نه نونی نه یه تیکه پیازی کیفیت میتونست خیلی بیشتر باشه,SAD
13601,سالاد یخ زده بود و پنیر روی سالاد مونده بود,سالاد یخ‌زده پنیر سالاد مونده,SAD
3271,ساندویچ خیلی کم حجم و کوچیک بود,ساندویچ خیلی کم‌حجم کوچیک,HAPPY
12647,هزینه پیک رستوران زیاد است!,هزینه پیک رستوران زیاد,HAPPY
13931,تشکر برای ارسال سریع ومطلوب خداقوت,تشکر ارسال سریع ومطلوب خداقوت,HAPPY


## 6. Check again: empty texts, duplicates, conflicts
After normalization, two reviews that looked different may now be identical.

In [13]:
n0 = len(df)

empty = df["clean_text"].str.strip() == ""
print("empty after cleaning:", empty.sum())
df = df[~empty]

conflict = df.groupby("clean_text")["sentiment"].nunique()
conflict = conflict[conflict > 1].index
print("texts with conflicting labels:", len(conflict),
      "| rows:", df["clean_text"].isin(conflict).sum())
df = df[~df["clean_text"].isin(conflict)]

dups = df.duplicated(subset="clean_text").sum()
print("duplicates after cleaning:", dups)
df = df.drop_duplicates(subset="clean_text").reset_index(drop=True)

print(f"\nrows: {n0:,} -> {len(df):,}")
print(df["label"].value_counts())

empty after cleaning: 2
texts with conflicting labels: 57 | rows: 149
duplicates after cleaning: 749

rows: 70,000 -> 69,100
label
SAD      34764
HAPPY    34336
Name: count, dtype: int64


In [14]:
# what did the conflicting ones look like?
raw_conf = raw.assign(clean_text=raw["comment"].apply(clean_text))
raw_conf = raw_conf[raw_conf["clean_text"].isin(conflict)].sort_values("clean_text")
raw_conf[["comment", "label", "clean_text"]].head(10)

,comment,label,clean_text
153,اصلا مثل قبل نبود,HAPPY,اصلا نبود
39882,اصلا مورد پ-ن- نبود,SAD,اصلا نبود
52740,بستنی‌ها کاملا آب شده بود!!!!,SAD,بستنی‌ها کاملا آب
2051,بستنی‌ها کاملا آب شده بودن!!,HAPPY,بستنی‌ها کاملا آب
37946,بیشتر تخم مرغ‌ها شکسته بود,SAD,بیشتر تخم مرغ‌ها شکسته
68659,بیشتر تخم مرغ‌ها شکسته بودن,HAPPY,بیشتر تخم مرغ‌ها شکسته
57081,تخم مرغها شکسته بودن,SAD,تخم مرغها شکسته
44459,تا از تخم مرغها شکسته بود,HAPPY,تخم مرغها شکسته
52258,تخم مرغ‌ها شکسته بود: /,SAD,تخم مرغ‌ها شکسته
56358,همه تخم مرغ‌ها شکسته,SAD,تخم مرغ‌ها شکسته


## 7. Frequent words and word pairs after cleaning

In [15]:
def top_tokens(texts, n=20):
    c = Counter(t for txt in texts for t in txt.split())
    return pd.Series([w for w, _ in c.most_common(n)])

def top_bigrams(texts, n=20):
    c = Counter()
    for txt in texts:
        toks = txt.split()
        c.update(" ".join(p) for p in zip(toks, toks[1:]))
    return pd.Series([w for w, _ in c.most_common(n)])

happy = df.loc[df["label"] == "HAPPY", "clean_text"]
sad = df.loc[df["label"] == "SAD", "clean_text"]

pd.DataFrame({
    "HAPPY words": top_tokens(happy), "SAD words": top_tokens(sad),
    "HAPPY pairs": top_bigrams(happy), "SAD pairs": top_bigrams(sad),
})

,HAPPY words,SAD words,HAPPY pairs,SAD pairs
0,عالی,خیلی,خیلی خوب,سفارش دادم
1,خیلی,غذا,دستم رسید,خوب نبود
2,خوب,سفارش,خوب ولی,خیلی بد
3,غذا,کیفیت,اسنپ فود,غذا سرد
4,کیفیت,نبود,بسیار عالی,کیفیت غذا
5,ولی,اصلا,همیشه عالی,اسنپ فود
6,خوشمزه,سرد,کیفیت غذا,کاملا سرد
7,رسید,پیتزا,خیلی خوشمزه,خیلی خیلی
8,ممنون,ولی,راضی بودم,سفارش بودم
9,بسیار,بسیار,خیلی عالی,تازه نبود


In [16]:
# the negation question from notebook 01: where does «خوب» appear in SAD reviews?
sad_good = df[(df["label"] == "SAD") & df["clean_text"].str.contains(r"\bخوب\b", regex=True)]
print("SAD reviews containing «خوب»:", len(sad_good))
print("...followed by «نبود»:", sad_good["clean_text"].str.contains("خوب نبود").sum())
sad_good[["comment", "clean_text"]].head(8)

SAD reviews containing «خوب»: 3477
...followed by «نبود»: 1142


,comment,clean_text
9,همه چی خوب ولی هات داگ دورش کلا سوخته بود و داخلش خام بود!!!!,چی خوب ولی هات داگ دورش کلا سوخته داخلش خام
28,بد نبود اما برخلاف دفعات قبل چرب‌تر بود و بال هام خوب نپخته بود,بد نبود اما برخلاف دفعات چرب‌تر بال هام خوب نپخته
55,باروژ باروژ این بود …!!! من سه مدل سیر و استیک. استیک و پیتزا باروژ گرفتم باید به باروژ تبریک بگم. خوب بلید جیب مردم و خالی کنید با این قیمت‌های ب...,باروژ باروژ مدل سیر استیک استیک پیتزا باروژ گرفتم باروژ تبریک بگم خوب بلید جیب مردم خالی قیمت‌های بالاتون باری باروژ خرید آبروم جلو همکارام بردین ...
65,مرغ خوب سرخ شده بود ولی محتویات شکم زیاد جالب نبود …,مرغ خوب سرخ‌شده ولی محتویات شکم زیاد جالب نبود
80,جوجه کباب اصلا خوب نبود. اما خوراک ماهی بد نبود,جوجه‌کباب اصلا خوب نبود اما خوراک ماهی بد نبود
81,برخورد خوب و مناسب پیک,برخورد خوب مناسب پیک
97,کیفیت خوب بود اما غذا هم سرد بود وهم حجمش بسیارکوچیکه و سیر نمیشی بایدونه,کیفیت خوب اما غذا سرد وهم حجمش بسیارکوچیکه سیر نمیشی بایدونه
118,حجم و کیفیت برگر خوب بود ولی اصلا پنیر نداشت!، خیارشور هم کم گذاشته بودن,حجم کیفیت برگر خوب ولی اصلا پنیر نداشت خیارشور کم گذاشته


## 8. Train / validation / test split
We split **once** and save the files, so every model in the next notebooks is trained and tested on exactly the same data.
- train 80% — to fit models
- validation 10% — to tune settings
- test 10% — touched only once, at the very end

`stratify` keeps the 50/50 label balance in each part.

In [17]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["sentiment"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["sentiment"], random_state=42)

cols = ["comment", "clean_text", "label", "sentiment"]
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part[cols].to_csv(f"{OUT_DIR}/{name}.csv", index=False)
    print(f"{name:5} {len(part):6,} rows | positive share: {part['sentiment'].mean():.3f}")

print("\nSaved to:", OUT_DIR)

train 55,280 rows | positive share: 0.497
val    6,910 rows | positive share: 0.497
test   6,910 rows | positive share: 0.497

Saved to: /content/drive/MyDrive/snappfood-sentiment/processed


## 9. Key findings
- **Less noise than expected:** no URLs, no Arabic ي/ك and no emojis. The dataset creator had already removed them. About 9.5% of reviews contain digits and 6.9% contain repeated characters (e.g. «عالیییی», «!!!»).
- **The stopword trap:** hazm's default stopword list is built for news text and includes key sentiment words such as «عالی», «خوب», «نبود», «نه» and «متاسفانه». Using it as-is would erase the strongest signals, so a custom KEEP list of ~45 negation, contrast, intensity and evaluation words was added back.
- **Hidden duplicates:** zero exact duplicates before cleaning, but 749 after normalization. Removing them prevents train/test leakage. 70,000 -> 69,100 rows, still balanced (49.7% positive).
- **Label noise:** some clearly negative reviews are labeled HAPPY (e.g. «بستنی‌ها کاملاً آب شده بود»). Labels probably come from star ratings, so no model can reach 100% accuracy.
- **Stopword removal has a cost:** in a few cases different sentences collapse into the same text (e.g. «اصلا مثل قبل نبود» -> «اصلا نبود»). Only ~0.2% of rows were affected. Lighter cleaning will be considered for the LSTM model.
- **Negation and contrast confirmed:** «خوب نبود» is the 2nd most common word pair in SAD reviews. About 1/3 of SAD reviews containing «خوب» use «خوب نبود», and many others follow a «X was good, but Y was bad» pattern. Bigrams will be used in modeling.
- **Fixed split:** train 55,280 / validation 6,910 / test 6,910, stratified (49.7% positive in each), saved for all later models.